# Tutorial — comparing inner solvers (Adam / SGD / CG / L-BFGS) on 20-cell meshes

This notebook builds **5 random 20-cell bounded meshes** and relaxes each one with four
inner-loop solvers, then plots the results side by side using VertAX's plotting functions.

**Solvers compared**
- **Adam**   — `optax.adam`, adaptive first-order.
- **SGD**    — `optax.sgd`, plain gradient descent.
- **CG**     — `vertax.nonlinear_cg`, nonlinear conjugate gradient (Polak–Ribière +
  restarts) with a zoom line search.
- **L-BFGS** — `optax.lbfgs`, limited-memory quasi-Newton with a zoom line search.

All four minimise the **same** energy (`vertax.energy.energy_bounded`) through the same
`BoundedBilevelOptimizer.inner_optimization` loop, with T1 transitions enabled. Only the
`inner_solver` changes.

**Layout** — one row per mesh, five columns:

| Initial | Adam | SGD | CG | L-BFGS |
|---------|------|-----|----|--------|

Each panel shows the mesh coloured by **tension** (edges) and **area** (cells), and its
title reports the **solver**, the **number of inner iterations**, the **energy**, and the
energy split into its **tension** and **area** parts.

## How the nonlinear conjugate gradient solver works

`vertax.nonlinear_cg` is a **Polak–Ribière** nonlinear conjugate gradient (CG) method
with automatic restarts and a zoom line search. It is a drop-in `optax` solver: it plugs
into the same loops as `optax.adam` / `optax.sgd`, provided the loop forwards the
line-search arguments `value`, `grad` and `value_fn` to `solver.update`.

### Why conjugate gradient
Steepest descent ($d = -\nabla f$) **zig-zags** in elongated valleys: each step partly
undoes the previous one. CG removes this by making successive search directions
*conjugate* (non-interfering with respect to the local curvature), so progress along a
valley is not destroyed at the next step. On a strictly **quadratic** objective in $n$
variables it converges in at most $n$ steps; a general energy is not quadratic, hence
*nonlinear* CG with periodic restarts.

### The algorithm (one `update` call)
Given the gradient $g_k = \nabla f(x_k)$, the previous gradient $g_{k-1}$ and the previous
direction $d_{k-1}$:

1. **Polak–Ribière coefficient**

   $$\beta = \frac{\langle g_k,\; g_k - g_{k-1}\rangle}{\lVert g_{k-1}\rVert^2}.$$

   The $g_k - g_{k-1}$ term injects curvature information **without ever forming a
   Hessian**.

2. **Restart** (set $\beta = 0 \Rightarrow$ fall back to $-g$) when either
   - it has been `restart_every` steps since the last restart (conjugacy degrades on a
     non-quadratic objective), or
   - $\beta \le 0$ — the **PR+** safeguard $\beta^+ = \max(0, \beta)$: a negative $\beta$
     means the old direction is no longer useful, so we drop it.

3. **Direction** $\;d_k = -g_k + \beta\, d_{k-1}\;$
   ($\beta = 0 \Rightarrow$ steepest descent; $\beta > 0 \Rightarrow$ curve the search
   along the valley).

4. **Descent guard**: a line search is only valid if $d$ goes downhill, i.e.
   $\langle d, g\rangle < 0$. Rounding / non-quadratic effects can break this; if so we
   fall back to the always-valid steepest-descent direction $-g$.

5. **Zoom line search** along $d$: chooses the step size satisfying the (strong) Wolfe
   conditions by re-evaluating $f$ along the line. `update` returns $\text{step}\cdot d$ as
   the optax updates.

### Line-search requirement
Like `optax.lbfgs`, this solver wraps `optax.scale_by_zoom_linesearch` and therefore needs
the loss value and a callable to re-evaluate it. The enclosing loop must call:

```python
updates, state = solver.update(grad, state, params,
                               value=current_value, grad=grad,
                               value_fn=loss_fn)
```

where `loss_fn(params) -> scalar`. `optax.adam` / `optax.sgd` **ignore** these extra
arguments, so a loop that always forwards them stays compatible with both.

## How the L-BFGS solver works

`optax.lbfgs` is a **quasi-Newton** method. Like CG it is a line-search solver (it needs
the same `value`, `grad`, `value_fn` arguments), but instead of building *conjugate
directions* it builds an **approximation of the inverse Hessian** from the gradients seen
so far, and multiplies the gradient by it.

### Why quasi-Newton
A pure Newton step is $d = -H^{-1} g$, where $H = \nabla^2 f$ is the Hessian: it rescales
the gradient by the local curvature, so it goes *straight* to the minimum of a quadratic
in one step. But forming and inverting $H$ (size $p \times p$ for $p$ parameters) is
expensive. **BFGS** avoids the explicit Hessian by updating an inverse-Hessian estimate
from successive gradient differences. **L-BFGS** (*limited-memory*) goes further: it never
stores the $p \times p$ matrix at all — it keeps only the last `memory_size` pairs

$$s_k = x_{k} - x_{k-1}, \qquad y_k = g_{k} - g_{k-1},$$

and reconstructs the action of $H^{-1}$ on the gradient from them via the **two-loop
recursion**. Memory and cost per step are $O(p \cdot m)$ with $m=$ `memory_size`, not
$O(p^2)$.

### The algorithm (one `update` call)
1. **Curvature pairs.** Record $s_k = x_k - x_{k-1}$ and $y_k = g_k - g_{k-1}$; drop the
   oldest pair if more than `memory_size` are stored.
2. **Two-loop recursion.** Apply the implicit inverse-Hessian estimate to the current
   gradient using the stored $(s_i, y_i)$ pairs, producing a search direction
   $d_k \approx -H^{-1} g_k$. An initial scaling $\gamma_k = \frac{\langle s_k, y_k\rangle}{\langle y_k, y_k\rangle}$
   (enabled by `scale_init_precond`) sets the scale of the first guess.
3. **Zoom line search** along $d_k$: the (strong) Wolfe step size, re-evaluating $f$ along
   the line — exactly the line search CG also uses.

### CG vs L-BFGS
- **CG** carries *one* previous direction and a scalar $\beta$ — almost no memory, cheap.
- **L-BFGS** carries the last `memory_size` curvature pairs — more memory, but a richer
  curvature model, so it usually needs fewer iterations on smooth problems.

Both are drop-in `optax` solvers here: the inner loop forwards `value`/`grad`/`value_fn`
unconditionally, so swapping `inner_solver` between `optax.adam`, `optax.sgd`,
`vertax.nonlinear_cg` and `optax.lbfgs` is the only change.

In [1]:
import os, math
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import jax
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

from vertax import (
    BoundedMesh, BoundedBilevelOptimizer, nonlinear_cg,
    get_plot_mesh, EdgePlot, FacePlot, VertexPlot,
)
from vertax.energy import energy_bounded
from vertax.geo import get_area_bounded, get_edge_length, get_surface_length

# ── Experiment settings ──────────────────────────────────────────────────────
N_CELLS   = 20
N_MESHES  = 5
SEEDS     = list(range(N_MESHES))
MAX_ITERS = 100          # upper bound on inner iterations (early stopping may stop sooner)
MIN_DIST_T1 = 0.005
TARGET_AREA = 0.6        # must match energy_bounded's TARGET_AREA

print("JAX devices:", jax.devices())

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


JAX devices: [CpuDevice(id=0)]


## Helpers

- `make_mesh(seed)` — a random 20-cell mesh with random edge tensions and a uniform
  target area, exactly as in the bounded tutorial.
- `metrics(mesh)` — total energy and its decomposition. `energy_bounded` is
  `20·Σ(area−target)² + Σ length·tension`, so we report:
  - **E**            : total energy,
  - **E_tension**    : `Σ length·tension` (inner + surface edges),
  - **E_area**       : `20·Σ(area−target)²`,
  - plus mean cell area and mean edge tension for context.
- `relax(seed, solver)` — run `inner_optimization` with the given solver and return the
  relaxed mesh and the number of iterations actually performed.

In [ ]:
def make_mesh(seed: int) -> BoundedMesh:
    # 20-cell bounded mesh with random tensions and uniform target area.
    w = math.sqrt(N_CELLS)
    m = BoundedMesh.from_random_seeds(nb_seeds=N_CELLS, width=w, height=w, random_key=seed)
    m.vertices_params = jnp.asarray([0.0] * m.nb_vertices)
    key = jax.random.PRNGKey(seed)
    # Random per-edge tension in ~[1, 2] (same convention as the bounded tutorial).
    m.edges_params = 1 + jax.nn.sigmoid(jax.random.uniform(key, (m.nb_edges,)) * 20 - 10)
    m.faces_params = jnp.asarray([TARGET_AREA] * m.nb_faces)
    return m


def metrics(m: BoundedMesh) -> dict:
    # Total energy + tension/area decomposition + mean area / mean tension.
    vt = jnp.array(m.vertices); at = jnp.array(m.angles)
    he = jnp.array(m.edges);    fc = jnp.array(m.faces)
    vp = jnp.array(m.vertices_params)
    ep = jnp.array(m.edges_params); fp = jnp.array(m.faces_params)

    E_total = float(energy_bounded(vt, at, he, fc, None, None, None, vp, ep, fp))

    # Rebuild the full tables the way energy_bounded does internally.
    vt_full = jnp.vstack([jnp.array([[0.0, 0.0], [1.0, 1.0]]), vt])
    at_full = jnp.repeat(at, 2)
    tensions = jax.nn.sigmoid(ep) + 1          # energy_bounded applies sigmoid+1 to he_params

    n_edges = at.size
    unique_edges = jnp.arange(n_edges) * 2
    all_edges = jnp.arange(n_edges * 2)

    inner = jnp.sum(jax.vmap(lambda e, t: get_edge_length(e, vt_full, he) * t)(
        unique_edges, tensions))
    surface = jnp.sum(jax.vmap(lambda e, t: get_surface_length(e, vt_full, at_full, he) * t)(
        all_edges, jnp.repeat(tensions, 2)))
    E_tension = float(inner + surface)

    areas = jax.vmap(lambda f: get_area_bounded(f, vt_full, at_full, he, fc))(
        jnp.arange(fc.shape[0]))
    E_area = float(20.0 * jnp.sum((areas - fp[0]) ** 2))   # 20 = K_areas in energy_bounded

    return dict(E=E_total, E_tension=E_tension, E_area=E_area,
                mean_area=float(jnp.mean(areas)), mean_tension=float(jnp.mean(tensions)))


def relax(seed: int, solver: optax.GradientTransformation):
    # Relax a fresh mesh with the given inner solver; return (mesh, n_iterations).
    m = make_mesh(seed)
    opt = BoundedBilevelOptimizer()
    opt.loss_function_inner = energy_bounded
    opt.inner_solver        = solver
    opt.update_T1           = True
    opt.min_dist_T1         = MIN_DIST_T1
    opt.max_nb_iterations   = MAX_ITERS
    opt.tolerance           = 1e-6
    opt.patience            = 20
    loss_history = opt.inner_optimization(m)
    return m, len(loss_history)


# One factory per column so every mesh gets a fresh solver state.
SOLVERS = {
    "Adam":   lambda: optax.adam(learning_rate=1e-3),
    "SGD":    lambda: optax.sgd(learning_rate=1e-2),
    "CG":     lambda: nonlinear_cg(restart_every=10),
    "L-BFGS": lambda: optax.lbfgs(),
}

## Run the four solvers on all 5 meshes and plot

For each mesh: render the initial state, then relax with Adam, SGD, CG and L-BFGS and
render each result. Panels are coloured by tension (edges, `coolwarm`) and area (cells,
`cividis`). The energy table is printed below the figure.

In [ ]:
PLOT_KW = dict(
    edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name="tension",
    face_plot=FacePlot.AREA, face_parameters_name="area",
    vertex_plot=VertexPlot.INVISIBLE,
)


def render(mesh, title, ax):
    # Render a mesh into an existing axis via get_plot_mesh + imshow.
    # The title is set on `ax` (set_title) only; passing it to get_plot_mesh too
    # would bake a second copy into the image and overlap the two.
    fig_m, _ = get_plot_mesh(mesh, **PLOT_KW)
    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
    img = np.roll(img, -1, axis=2)   # ARGB -> RGBA
    plt.close(fig_m)
    ax.imshow(img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


N_COLS = 1 + len(SOLVERS)   # Initial + one column per solver (Adam, SGD, CG, L-BFGS)
fig, axes = plt.subplots(N_MESHES, N_COLS, figsize=(5 * N_COLS, 5 * N_MESHES),
                         constrained_layout=True)
if N_MESHES == 1:
    axes = axes[None, :]

header = f'{"mesh":<6} {"solver":<7} {"iters":>6} {"E":>10} {"E_tension":>11} {"E_area":>9} {"meanArea":>9} {"meanTens":>9}'
print(header)
print("-" * len(header))

for row, seed in enumerate(SEEDS):
    # Column 0 — initial mesh
    m_init = make_mesh(seed)
    mi = metrics(m_init)
    render(m_init, f"seed {seed} — Initial\nE={mi['E']:.1f}  tens={mi['E_tension']:.1f}  area={mi['E_area']:.1f}",
           axes[row, 0])
    print(f'{seed:<6} {"init":<7} {"-":>6} {mi["E"]:>10.2f} {mi["E_tension"]:>11.2f} '
          f'{mi["E_area"]:>9.2f} {mi["mean_area"]:>9.3f} {mi["mean_tension"]:>9.3f}')

    # Columns 1..N — one solver each
    for col, (name, make_solver) in enumerate(SOLVERS.items(), start=1):
        m, n_it = relax(seed, make_solver())
        mm = metrics(m)
        title = (f"seed {seed} — {name} ({n_it} it)\n"
                 f"E={mm['E']:.1f}  tens={mm['E_tension']:.1f}  area={mm['E_area']:.1f}")
        render(m, title, axes[row, col])
        print(f'{seed:<6} {name:<7} {n_it:>6} {mm["E"]:>10.2f} {mm["E_tension"]:>11.2f} '
              f'{mm["E_area"]:>9.2f} {mm["mean_area"]:>9.3f} {mm["mean_tension"]:>9.3f}')

fig.suptitle("20-cell meshes relaxed by Adam / SGD / CG / L-BFGS (colour: tension on edges, area on cells)",
             fontsize=14)
plt.show()

## Notes

- **Same energy, same loop, only the solver differs.** Adam/SGD ignore the line-search
  arguments that the inner loop now forwards; CG uses them. This is what makes
  `nonlinear_cg` a drop-in replacement for `optax` first-order solvers here.
- **Energy decomposition.** `E = E_tension + E_area` where
  `E_tension = Σ length·tension` and `E_area = 20·Σ(area − target)²`. Watching the split
  shows whether a solver is shrinking edges (tension) or homogenising cell areas.
- **Iteration counts** can differ between solvers because early stopping (`tolerance`,
  `patience`) triggers at different points; CG/line-search steps also do extra internal
  energy evaluations not counted here.
- The same target area (`TARGET_AREA = 0.6`) and the same random tensions are used for all
  solvers on a given mesh, so the comparison is fair.